In [1]:
# ============================================================
# CELL 1: Imports and Data Loading
# ============================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef, 
                              roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier
import time
import os
import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DATA_PATH = r"..\..\Data\Main\modelling_dataset.csv"
OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
model_df = df.dropna(subset=['volatility_20d', 'next_day_return']).copy()
model_df = model_df.sort_values(['ticker', 'date']).reset_index(drop=True)
model_df['year_month'] = model_df['date'].dt.to_period('M')

print(f"Shape: {model_df.shape[0]:,} rows, {model_df['ticker'].nunique()} tickers")
print(f"Date range: {model_df['date'].min().date()} to {model_df['date'].max().date()}")

# Training period for tuning
train_full = model_df[model_df['date'] < '2022-01-01'].copy()
print(f"\nTraining period (2020-2021): {len(train_full):,} rows, {train_full['ticker'].nunique()} tickers")

Device: cpu
Shape: 578,903 rows, 523 tickers
Date range: 2020-01-31 to 2025-12-30

Training period (2020-2021): 165,150 rows, 472 tickers


In [2]:
# ============================================================
# CELL 2: Feature Sets, Model Definitions, Evaluation Helpers
# ============================================================

# --- Feature Sets (universal across all models) ---
FEATURE_SETS = {
    'A_sentiment_only':   ['sentiment'],
    'B_volatility_only':  ['volatility_20d'],
    'C_sent_plus_vol':    ['sentiment', 'volatility_20d'],
    'D_with_interaction': ['sentiment', 'volatility_20d', 'sent_x_vol'],
}
print("Feature sets:")
for name, cols in FEATURE_SETS.items():
    print(f"  {name}: {cols}")

# --- Evaluation ---
def evaluate(y_true, y_pred, y_prob=None):
    """Compute all evaluation metrics."""
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            metrics['AUC-ROC'] = roc_auc_score(y_true, y_prob)
        except:
            metrics['AUC-ROC'] = np.nan
    metrics['Pred_Up_Pct'] = y_pred.mean() if len(y_pred) > 0 else np.nan
    return metrics

# --- LSTM Model ---
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        return self.fc(last_output)

def create_sequences(data, features, seq_length):
    """Create LSTM sequences grouped by ticker."""
    all_X, all_y, all_dates, all_volatile = [], [], [], []
    for t in data['ticker'].unique():
        td = data[data['ticker'] == t].sort_values('date')
        fv = td[features].values
        tv = td['target'].values
        dv = td['date'].values
        vv = td['volatile_market'].values
        for j in range(seq_length, len(td)):
            all_X.append(fv[j - seq_length:j])
            all_y.append(tv[j])
            all_dates.append(dv[j])
            all_volatile.append(vv[j])
    return np.array(all_X), np.array(all_y), np.array(all_dates), np.array(all_volatile)

print("\nHelpers defined.")

Feature sets:
  A_sentiment_only: ['sentiment']
  B_volatility_only: ['volatility_20d']
  C_sent_plus_vol: ['sentiment', 'volatility_20d']
  D_with_interaction: ['sentiment', 'volatility_20d', 'sent_x_vol']

Helpers defined.


In [3]:
# ============================================================
# CELL 3: XGBoost Hyperparameter Grid Search
# ============================================================
# 
# DESIGN LOGIC:
# - Tuning is done on feature set D (all features) — the superset.
#   Best params are then applied to all 4 feature sets.
#   Rationale: tuning per feature set would be ideal but creates 
#   4× the computation and risks overfitting to specific feature combos.
#
# - Evaluation uses per-stock 70/30 time-series split within 2020-2021.
#   This mirrors the actual prediction setup (per-stock models).
#   The split is chronological, not random — no look-ahead bias.
#
# - Selection criterion: MCC (primary), with Accuracy reported.
#   MCC is preferred because near 50/50 class balance means a model
#   that always predicts "up" gets ~52% accuracy but MCC ≈ 0.
#
# - Grid design rationale:
#   n_estimators [50, 100, 200]: covers small to medium ensembles.
#     Financial data is noisy — very large ensembles risk overfitting.
#   max_depth [2, 3, 5]: shallow trees reduce overfitting on noisy data.
#     Depth 2-3 is typical for financial applications.
#   learning_rate [0.01, 0.05, 0.1]: standard range.
#     Lower rates learn more gradually but need more estimators.
#     0.01 with 200 trees and 0.1 with 50 trees test this trade-off.
# ============================================================

print("=" * 70)
print("XGBOOST HYPERPARAMETER TUNING")
print("=" * 70)

xgb_tune_features = FEATURE_SETS['C_sent_plus_vol']
print(f"Tuning on feature set C: {xgb_tune_features}")

xgb_param_grid = {
    'n_estimators':  [50, 100, 200],
    'max_depth':     [2, 3, 5],
    'learning_rate': [0.01, 0.05, 0.1],
}

total_combos = 27
tickers_tune = train_full['ticker'].unique()
print(f"Tickers for tuning: {len(tickers_tune)}")
print(f"Total combinations: {total_combos}\n")

print(f"{'#':<4} {'n_est':>6} {'depth':>6} {'lr':>6} {'Acc':>8} {'MCC':>8} {'F1':>8} {'Up%':>7} {'Time':>6}")
print("-" * 60)

xgb_grid_results = []
count = 0

for n_est in xgb_param_grid['n_estimators']:
    for depth in xgb_param_grid['max_depth']:
        for lr in xgb_param_grid['learning_rate']:
            count += 1
            t0 = time.time()
            
            all_y_true, all_y_pred = [], []
            
            for t in tickers_tune:
                t_data = train_full[train_full['ticker'] == t].sort_values('date')
                
                # Skip tickers with too little data
                if len(t_data) < 80:
                    continue
                
                # 70/30 chronological split
                split_idx = int(len(t_data) * 0.7)
                cv_train = t_data.iloc[:split_idx]
                cv_val = t_data.iloc[split_idx:]
                
                if len(cv_train) < 50 or len(cv_val) < 20:
                    continue
                
                # Standardise
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(cv_train[xgb_tune_features])
                X_va = scaler.transform(cv_val[xgb_tune_features])
                y_tr = cv_train['target'].values
                y_va = cv_val['target'].values
                
                n_pos = max(y_tr.sum(), 1)
                n_neg = max(len(y_tr) - n_pos, 1)
                
                xgb = XGBClassifier(
                    n_estimators=n_est, max_depth=depth, learning_rate=lr,
                    scale_pos_weight=n_neg / n_pos,
                    random_state=42, use_label_encoder=False,
                    eval_metric='logloss', verbosity=0
                )
                xgb.fit(X_tr, y_tr)
                y_pred = xgb.predict(X_va)
                
                all_y_true.extend(y_va)
                all_y_pred.extend(y_pred)
            
            all_y_true = np.array(all_y_true)
            all_y_pred = np.array(all_y_pred)
            
            acc = accuracy_score(all_y_true, all_y_pred)
            mcc = matthews_corrcoef(all_y_true, all_y_pred)
            f1 = f1_score(all_y_true, all_y_pred, zero_division=0)
            up_pct = all_y_pred.mean()
            elapsed = time.time() - t0
            
            xgb_grid_results.append({
                'n_estimators': n_est, 'max_depth': depth, 'learning_rate': lr,
                'accuracy': acc, 'mcc': mcc, 'f1': f1, 'pred_up_pct': up_pct
            })
            
            print(f"{count:<4} {n_est:>6} {depth:>6} {lr:>6.2f} {acc:>8.4f} {mcc:>8.4f} {f1:>8.4f} {up_pct:>6.1%} {elapsed:>5.0f}s")

xgb_grid_df = pd.DataFrame(xgb_grid_results)
print(f"\nAll {total_combos} combinations tested.")


XGBOOST HYPERPARAMETER TUNING
Tuning on feature set C: ['sentiment', 'volatility_20d']
Tickers for tuning: 472
Total combinations: 27

#     n_est  depth     lr      Acc      MCC       F1     Up%   Time
------------------------------------------------------------
1        50      2   0.01   0.5016   0.0024   0.5182  50.9%    10s
2        50      2   0.05   0.5010   0.0008   0.5185  51.1%    10s
3        50      2   0.10   0.5004   0.0002   0.5162  50.7%    10s
4        50      3   0.01   0.5014   0.0033   0.5116  49.5%    11s
5        50      3   0.05   0.5003   0.0006   0.5128  50.0%    11s
6        50      3   0.10   0.5028   0.0053   0.5167  50.3%    11s
7        50      5   0.01   0.5031   0.0056   0.5181  50.5%    12s
8        50      5   0.05   0.5033   0.0062   0.5173  50.3%    12s
9        50      5   0.10   0.5016   0.0027   0.5160  50.4%    12s
10      100      2   0.01   0.5011   0.0012   0.5188  51.1%    13s
11      100      2   0.05   0.5002  -0.0007   0.5180  51.1%    14s

In [5]:
# ============================================================
# CELL 4: XGBoost Grid Results Analysis
# ============================================================

print("=" * 70)
print("XGBOOST TUNING RESULTS")
print("=" * 70)

# Sort by MCC (primary criterion)
xgb_sorted = xgb_grid_df.sort_values('mcc', ascending=False).reset_index(drop=True)

print(f"\nTop 5 by MCC:")
print(f"{'Rank':<6} {'n_est':>6} {'depth':>6} {'lr':>6} {'Acc':>8} {'MCC':>8} {'F1':>8} {'Up%':>7}")
print("-" * 55)
for i, row in xgb_sorted.head(5).iterrows():
    print(f"{i+1:<6} {int(row['n_estimators']):>6} {int(row['max_depth']):>6} {row['learning_rate']:>6.2f} {row['accuracy']:>8.4f} {row['mcc']:>8.4f} {row['f1']:>8.4f} {row['pred_up_pct']:>6.1%}")

# Check spread — how sensitive are results to hyperparameters?
print(f"\nAccuracy range: {xgb_grid_df['accuracy'].min():.4f} to {xgb_grid_df['accuracy'].max():.4f} (spread: {xgb_grid_df['accuracy'].max()-xgb_grid_df['accuracy'].min():.4f})")
print(f"MCC range:      {xgb_grid_df['mcc'].min():.4f} to {xgb_grid_df['mcc'].max():.4f} (spread: {xgb_grid_df['mcc'].max()-xgb_grid_df['mcc'].min():.4f})")

# Select best
best_xgb = xgb_sorted.iloc[0]
XGB_BEST = {
    'n_estimators': int(best_xgb['n_estimators']),
    'max_depth': int(best_xgb['max_depth']),
    'learning_rate': best_xgb['learning_rate'],
}
print(f"\n=== SELECTED XGBoost PARAMETERS ===")
print(f"  n_estimators:  {XGB_BEST['n_estimators']}")
print(f"  max_depth:     {XGB_BEST['max_depth']}")
print(f"  learning_rate: {XGB_BEST['learning_rate']}")
print(f"  Validation MCC:      {best_xgb['mcc']:.4f}")
print(f"  Validation Accuracy: {best_xgb['accuracy']:.4f}")

# Save grid results
xgb_grid_df.to_csv(os.path.join(OUTPUT_DIR, "xgb_grid_search_results.csv"), index=False)
print(f"\nSaved: xgb_grid_search_results.csv")

XGBOOST TUNING RESULTS

Top 5 by MCC:
Rank    n_est  depth     lr      Acc      MCC       F1     Up%
-------------------------------------------------------
1          50      5   0.05   0.5033   0.0062   0.5173  50.3%
2          50      5   0.01   0.5031   0.0056   0.5181  50.5%
3          50      3   0.10   0.5028   0.0053   0.5167  50.3%
4         100      5   0.01   0.5026   0.0046   0.5178  50.6%
5         200      5   0.01   0.5024   0.0044   0.5173  50.5%

Accuracy range: 0.5002 to 0.5033 (spread: 0.0030)
MCC range:      -0.0007 to 0.0062 (spread: 0.0069)

=== SELECTED XGBoost PARAMETERS ===
  n_estimators:  50
  max_depth:     5
  learning_rate: 0.05
  Validation MCC:      0.0062
  Validation Accuracy: 0.5033

Saved: xgb_grid_search_results.csv


In [6]:
# ============================================================
# CELL 5: LSTM Hyperparameter Grid Search
# ============================================================
#
# DESIGN LOGIC:
# - Tuning on feature set C (all features), same rationale as XGBoost.
#
# - Temporal split within 2020-2021:
#     Train: Jan 2020 – Jun 2021 (18 months)
#     Validate: Jul 2021 – Dec 2021 (6 months)
#   This gives a meaningful validation period while preserving
#   sufficient training data. The split is at a fixed date,
#   not a percentage, because LSTM processes sequences and needs
#   a continuous chronological block for both train and validate.
#
# - Selection criterion: MCC among configurations with balanced
#   predictions (40-60% up). This two-step filter is necessary
#   because some LSTM configurations collapse to predicting
#   nearly all one class, which gives misleading accuracy (~52%)
#   but zero discriminative ability (MCC ≈ 0).
#
# - Grid design rationale:
#   hidden_size [32, 64, 128]: controls model capacity.
#     32 is minimal (prevents overfitting on noisy data).
#     128 tests whether more capacity helps.
#   epochs [5, 10, 15]: controls training duration.
#     Financial data is noisy — too many epochs = overfitting.
#     5 tests early stopping effect, 15 tests full convergence.
#   sequence_length [10, 20, 30]: how many trading days of history.
#     10 = ~2 weeks, 20 = ~1 month, 30 = ~6 weeks.
#     This tests whether longer memory helps in a noisy environment.
# ============================================================

print("=" * 70)
print("LSTM HYPERPARAMETER TUNING")
print("=" * 70)

lstm_tune_features = FEATURE_SETS['C_sent_plus_vol']
print(f"Tuning on feature set C: {lstm_tune_features}")

# Temporal split
lstm_train_period = model_df[model_df['date'] < '2021-07-01'].copy()
lstm_val_period = model_df[(model_df['date'] >= '2021-07-01') & (model_df['date'] < '2022-01-01')].copy()

print(f"Train: {lstm_train_period['date'].min().date()} to {lstm_train_period['date'].max().date()} ({len(lstm_train_period):,} rows)")
print(f"Validate: {lstm_val_period['date'].min().date()} to {lstm_val_period['date'].max().date()} ({len(lstm_val_period):,} rows)")

lstm_param_grid = {
    'hidden_size':    [32, 64, 128],
    'epochs':         [5, 10, 15],
    'sequence_length': [10, 20, 30],
}

total_combos = 27
print(f"Total combinations: {total_combos}\n")

print(f"{'#':<4} {'hidden':>7} {'epochs':>7} {'seqlen':>7} {'Acc':>8} {'MCC':>8} {'F1':>8} {'Up%':>7} {'Time':>6}")
print("-" * 65)

lstm_grid_results = []
count = 0

for hs in lstm_param_grid['hidden_size']:
    for ep in lstm_param_grid['epochs']:
        for sl in lstm_param_grid['sequence_length']:
            count += 1
            t0 = time.time()
            
            # Build sequences
            X_train, y_train, _, _ = create_sequences(lstm_train_period, lstm_tune_features, sl)
            X_val, y_val, _, _ = create_sequences(lstm_val_period, lstm_tune_features, sl)
            
            if len(X_val) == 0 or len(X_train) == 0:
                print(f"{count:<4} {hs:>7} {ep:>7} {sl:>7} — SKIPPED (no data)")
                continue
            
            # Standardise
            scaler = StandardScaler()
            n_tr, seql, nf = X_train.shape
            X_train_s = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(n_tr, seql, nf)
            X_val_s = scaler.transform(X_val.reshape(-1, nf)).reshape(len(X_val), seql, nf)
            
            # Class weighting
            n_pos = max(y_train.sum(), 1)
            n_neg = max(len(y_train) - n_pos, 1)
            pos_weight = torch.FloatTensor([n_neg / n_pos]).to(device)
            
            # Build model
            model = LSTMModel(input_size=len(lstm_tune_features), hidden_size=hs, num_layers=1).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            # Train
            X_tr_t = torch.FloatTensor(X_train_s).to(device)
            y_tr_t = torch.FloatTensor(y_train).to(device)
            loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=1024, shuffle=True)
            
            model.train()
            for epoch in range(ep):
                for batch_X, batch_y in loader:
                    optimizer.zero_grad()
                    output = model(batch_X).squeeze()
                    loss = criterion(output, batch_y)
                    loss.backward()
                    optimizer.step()
            
            # Predict
            model.eval()
            X_val_t = torch.FloatTensor(X_val_s).to(device)
            with torch.no_grad():
                logits = model(X_val_t).squeeze().cpu().numpy()
                probs = 1 / (1 + np.exp(-logits))
            
            preds = (probs > 0.5).astype(int)
            acc = accuracy_score(y_val, preds)
            mcc = matthews_corrcoef(y_val, preds)
            f1 = f1_score(y_val, preds, zero_division=0)
            up_pct = preds.mean()
            elapsed = time.time() - t0
            
            lstm_grid_results.append({
                'hidden_size': hs, 'epochs': ep, 'sequence_length': sl,
                'accuracy': acc, 'mcc': mcc, 'f1': f1, 'pred_up_pct': up_pct
            })
            
            print(f"{count:<4} {hs:>7} {ep:>7} {sl:>7} {acc:>8.4f} {mcc:>8.4f} {f1:>8.4f} {up_pct:>6.1%} {elapsed:>5.0f}s")

lstm_grid_df = pd.DataFrame(lstm_grid_results)
print(f"\nAll {len(lstm_grid_results)} combinations tested.")

LSTM HYPERPARAMETER TUNING
Tuning on feature set C: ['sentiment', 'volatility_20d']
Train: 2020-01-31 to 2021-06-29 (116,354 rows)
Validate: 2021-07-01 to 2021-12-31 (48,796 rows)
Total combinations: 27

#     hidden  epochs  seqlen      Acc      MCC       F1     Up%   Time
-----------------------------------------------------------------
1         32       5      10   0.4705  -0.0046   0.0746   4.1%    12s
2         32       5      20   0.5284   0.0017   0.6871  97.7%    13s
3         32       5      30   0.4767  -0.0212   0.3356  25.8%    15s
4         32      10      10   0.4730  -0.0216   0.2849  20.6%    18s
5         32      10      20   0.5094  -0.0133   0.6184  75.6%    22s
6         32      10      30   0.4979  -0.0103   0.5359  55.3%    26s
7         32      15      10   0.4759  -0.0217   0.3433  26.7%    24s
8         32      15      20   0.4790  -0.0162   0.3452  26.6%    31s
9         32      15      30   0.4895  -0.0146   0.4757  44.5%    38s
10        64       5      10 

In [7]:
# ============================================================
# CELL 6: LSTM Grid Results Analysis
# ============================================================

print("=" * 70)
print("LSTM TUNING RESULTS")
print("=" * 70)

# Step 1: Filter for balanced predictions (40-60% up)
balanced = lstm_grid_df[(lstm_grid_df['pred_up_pct'] >= 0.40) & (lstm_grid_df['pred_up_pct'] <= 0.60)].copy()
degenerate = lstm_grid_df[(lstm_grid_df['pred_up_pct'] < 0.40) | (lstm_grid_df['pred_up_pct'] > 0.60)]

print(f"\nTotal configurations: {len(lstm_grid_df)}")
print(f"Balanced predictions (40-60% up): {len(balanced)}")
print(f"Degenerate predictions (outside 40-60%): {len(degenerate)}")

if len(degenerate) > 0:
    print(f"\nDegenerate configurations:")
    print(f"{'hidden':>7} {'epochs':>7} {'seqlen':>7} {'Acc':>8} {'MCC':>8} {'Up%':>7}")
    print("-" * 50)
    for _, row in degenerate.iterrows():
        print(f"{int(row['hidden_size']):>7} {int(row['epochs']):>7} {int(row['sequence_length']):>7} {row['accuracy']:>8.4f} {row['mcc']:>8.4f} {row['pred_up_pct']:>6.1%}")

# Step 2: Sort balanced configurations by MCC
if len(balanced) > 0:
    lstm_sorted = balanced.sort_values('mcc', ascending=False).reset_index(drop=True)
    
    print(f"\nTop 5 balanced configurations by MCC:")
    print(f"{'Rank':<6} {'hidden':>7} {'epochs':>7} {'seqlen':>7} {'Acc':>8} {'MCC':>8} {'F1':>8} {'Up%':>7}")
    print("-" * 60)
    for i, (_, row) in enumerate(lstm_sorted.head(5).iterrows()):
        print(f"{i+1:<6} {int(row['hidden_size']):>7} {int(row['epochs']):>7} {int(row['sequence_length']):>7} {row['accuracy']:>8.4f} {row['mcc']:>8.4f} {row['f1']:>8.4f} {row['pred_up_pct']:>6.1%}")
    
    best_lstm = lstm_sorted.iloc[0]
    LSTM_BEST = {
        'hidden_size': int(best_lstm['hidden_size']),
        'epochs': int(best_lstm['epochs']),
        'sequence_length': int(best_lstm['sequence_length']),
    }
else:
    print("\nWARNING: No balanced configurations found. Using defaults.")
    LSTM_BEST = {'hidden_size': 32, 'epochs': 10, 'sequence_length': 20}

print(f"\n=== SELECTED LSTM PARAMETERS ===")
print(f"  hidden_size:     {LSTM_BEST['hidden_size']}")
print(f"  epochs:          {LSTM_BEST['epochs']}")
print(f"  sequence_length: {LSTM_BEST['sequence_length']}")
if len(balanced) > 0:
    print(f"  Validation MCC:      {best_lstm['mcc']:.4f}")
    print(f"  Validation Accuracy: {best_lstm['accuracy']:.4f}")

# Check spread
print(f"\nAmong balanced configurations:")
print(f"  Accuracy range: {balanced['accuracy'].min():.4f} to {balanced['accuracy'].max():.4f} (spread: {balanced['accuracy'].max()-balanced['accuracy'].min():.4f})")
print(f"  MCC range:      {balanced['mcc'].min():.4f} to {balanced['mcc'].max():.4f} (spread: {balanced['mcc'].max()-balanced['mcc'].min():.4f})")

# Save
lstm_grid_df.to_csv(os.path.join(OUTPUT_DIR, "lstm_grid_search_results.csv"), index=False)
print(f"\nSaved: lstm_grid_search_results.csv")

LSTM TUNING RESULTS

Total configurations: 27
Balanced predictions (40-60% up): 8
Degenerate predictions (outside 40-60%): 19

Degenerate configurations:
 hidden  epochs  seqlen      Acc      MCC     Up%
--------------------------------------------------
     32       5      10   0.4705  -0.0046   4.1%
     32       5      20   0.5284   0.0017  97.7%
     32       5      30   0.4767  -0.0212  25.8%
     32      10      10   0.4730  -0.0216  20.6%
     32      10      20   0.5094  -0.0133  75.6%
     32      15      10   0.4759  -0.0217  26.7%
     32      15      20   0.4790  -0.0162  26.6%
     64       5      10   0.5123  -0.0071  74.6%
     64       5      20   0.4802   0.0053  13.5%
     64       5      30   0.4909  -0.0008  34.9%
     64      10      10   0.4769  -0.0049  15.8%
     64      10      20   0.5037  -0.0184  70.5%
     64      15      10   0.4814  -0.0184  34.2%
     64      15      30   0.5038  -0.0102  64.9%
    128       5      10   0.4823  -0.0195  36.5%
    128   

In [8]:
# ============================================================
# CELL 7: Walk-Forward Setup
# ============================================================

print("=" * 70)
print("WALK-FORWARD PREDICTION SETUP")
print("=" * 70)

all_months = sorted(model_df[model_df['date'] >= '2022-01-01']['year_month'].unique())
tickers = model_df['ticker'].unique()

print(f"Prediction period: {all_months[0]} to {all_months[-1]} ({len(all_months)} months)")
print(f"Tickers: {len(tickers)}")
print(f"\nXGBoost params: {XGB_BEST}")
print(f"LSTM params: {LSTM_BEST}")
print(f"\nFeature sets: {list(FEATURE_SETS.keys())}")
print(f"Total configurations: 3 models × 4 feature sets = 12")

# Master results storage
all_results = []

print("\nReady to run. Execute cells 8, 9, 10 in order.")

WALK-FORWARD PREDICTION SETUP
Prediction period: 2022-01 to 2025-12 (48 months)
Tickers: 523

XGBoost params: {'n_estimators': 50, 'max_depth': 5, 'learning_rate': np.float64(0.05)}
LSTM params: {'hidden_size': 128, 'epochs': 15, 'sequence_length': 30}

Feature sets: ['A_sentiment_only', 'B_volatility_only', 'C_sent_plus_vol', 'D_with_interaction']
Total configurations: 3 models × 4 feature sets = 12

Ready to run. Execute cells 8, 9, 10 in order.


In [9]:
# ============================================================
# CELL 8: Logistic Regression Walk-Forward (all 4 feature sets)
# Estimated time: ~5 minutes
# ============================================================

print("=" * 70)
print("LOGISTIC REGRESSION — WALK-FORWARD (POOLED)")
print("=" * 70)

for fs_name, fs_cols in FEATURE_SETS.items():
    print(f"\n  {fs_name}: {fs_cols}")
    
    y_true_all, y_pred_all, y_prob_all, y_vol_all = [], [], [], []
    t0 = time.time()
    
    for mi, month in enumerate(all_months):
        train = model_df[model_df['year_month'] < month]
        test = model_df[model_df['year_month'] == month]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[fs_cols])
        X_test = scaler.transform(test[fs_cols])
        
        lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        lr.fit(X_train, train['target'].values)
        
        y_pred = lr.predict(X_test)
        y_prob = lr.predict_proba(X_test)[:, 1]
        
        y_true_all.extend(test['target'].values)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)
        y_vol_all.extend(test['volatile_market'].values)
    
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    y_vol_all = np.array(y_vol_all)
    elapsed = time.time() - t0
    
    m_overall = evaluate(y_true_all, y_pred_all, y_prob_all)
    m_vol = evaluate(y_true_all[y_vol_all==1], y_pred_all[y_vol_all==1], y_prob_all[y_vol_all==1])
    m_calm = evaluate(y_true_all[y_vol_all==0], y_pred_all[y_vol_all==0], y_prob_all[y_vol_all==0])
    
    print(f"    Overall  — Acc: {m_overall['Accuracy']:.4f}, MCC: {m_overall['MCC']:.4f}, AUC: {m_overall['AUC-ROC']:.4f}")
    print(f"    Volatile — Acc: {m_vol['Accuracy']:.4f}, MCC: {m_vol['MCC']:.4f}, AUC: {m_vol['AUC-ROC']:.4f}")
    print(f"    Calm     — Acc: {m_calm['Accuracy']:.4f}, MCC: {m_calm['MCC']:.4f}, AUC: {m_calm['AUC-ROC']:.4f}")
    print(f"    Predictions: {len(y_true_all):,}, Up: {y_pred_all.mean():.1%} [{elapsed:.0f}s]")
    
    all_results.append({
        'model': 'Logistic Regression', 'feature_set': fs_name,
        **{f'overall_{k}': v for k, v in m_overall.items()},
        **{f'volatile_{k}': v for k, v in m_vol.items()},
        **{f'calm_{k}': v for k, v in m_calm.items()},
        'n_predictions': len(y_true_all), 'time_seconds': elapsed,
    })

print("\nLogistic Regression complete.")

LOGISTIC REGRESSION — WALK-FORWARD (POOLED)

  A_sentiment_only: ['sentiment']
    Overall  — Acc: 0.4993, MCC: 0.0016, AUC: 0.5005
    Volatile — Acc: 0.4973, MCC: -0.0037, AUC: 0.4985
    Calm     — Acc: 0.5009, MCC: 0.0045, AUC: 0.5019
    Predictions: 413,753, Up: 44.6% [5s]

  B_volatility_only: ['volatility_20d']
    Overall  — Acc: 0.4901, MCC: -0.0140, AUC: 0.4873
    Volatile — Acc: 0.4933, MCC: -0.0119, AUC: 0.4873
    Calm     — Acc: 0.4874, MCC: -0.0207, AUC: 0.4869
    Predictions: 413,753, Up: 39.0% [5s]

  C_sent_plus_vol: ['sentiment', 'volatility_20d']
    Overall  — Acc: 0.4972, MCC: -0.0042, AUC: 0.4968
    Volatile — Acc: 0.4954, MCC: -0.0090, AUC: 0.4959
    Calm     — Acc: 0.4986, MCC: -0.0012, AUC: 0.4975
    Predictions: 413,753, Up: 47.3% [5s]

  D_with_interaction: ['sentiment', 'volatility_20d', 'sent_x_vol']
    Overall  — Acc: 0.4969, MCC: -0.0049, AUC: 0.4962
    Volatile — Acc: 0.4951, MCC: -0.0096, AUC: 0.4952
    Calm     — Acc: 0.4984, MCC: -0.0019, AU

In [10]:
# ============================================================
# CELL 9: XGBoost Walk-Forward (all 4 feature sets, per-stock)
# Estimated time: ~2-3 hours
# ============================================================

print("=" * 70)
print("XGBOOST — WALK-FORWARD (PER-STOCK)")
print("=" * 70)

XGB_MIN_TRAIN = 100
XGB_MIN_TEST = 5

for fs_name, fs_cols in FEATURE_SETS.items():
    print(f"\n  {fs_name}: {fs_cols}")
    
    y_true_all, y_pred_all, y_prob_all, y_vol_all = [], [], [], []
    t0 = time.time()
    
    for mi, month in enumerate(all_months):
        train = model_df[model_df['year_month'] < month]
        test = model_df[model_df['year_month'] == month]
        
        for t in tickers:
            train_t = train[train['ticker'] == t]
            test_t = test[test['ticker'] == t]
            
            if len(train_t) < XGB_MIN_TRAIN or len(test_t) < XGB_MIN_TEST:
                continue
            
            scaler = StandardScaler()
            X_train = scaler.fit_transform(train_t[fs_cols])
            X_test = scaler.transform(test_t[fs_cols])
            y_train = train_t['target'].values
            y_test = test_t['target'].values
            
            n_pos = max(y_train.sum(), 1)
            n_neg = max(len(y_train) - n_pos, 1)
            
            xgb = XGBClassifier(
                n_estimators=XGB_BEST['n_estimators'],
                max_depth=XGB_BEST['max_depth'],
                learning_rate=XGB_BEST['learning_rate'],
                scale_pos_weight=n_neg / n_pos,
                random_state=42, use_label_encoder=False,
                eval_metric='logloss', verbosity=0
            )
            xgb.fit(X_train, y_train)
            
            y_pred = xgb.predict(X_test)
            y_prob = xgb.predict_proba(X_test)[:, 1]
            
            y_true_all.extend(y_test)
            y_pred_all.extend(y_pred)
            y_prob_all.extend(y_prob)
            y_vol_all.extend(test_t['volatile_market'].values)
        
        if (mi + 1) % 12 == 0:
            elapsed = time.time() - t0
            print(f"    Done {mi+1}/{len(all_months)} months [{elapsed:.0f}s]")
    
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    y_vol_all = np.array(y_vol_all)
    elapsed = time.time() - t0
    
    m_overall = evaluate(y_true_all, y_pred_all, y_prob_all)
    m_vol = evaluate(y_true_all[y_vol_all==1], y_pred_all[y_vol_all==1], y_prob_all[y_vol_all==1])
    m_calm = evaluate(y_true_all[y_vol_all==0], y_pred_all[y_vol_all==0], y_prob_all[y_vol_all==0])
    
    print(f"    Overall  — Acc: {m_overall['Accuracy']:.4f}, MCC: {m_overall['MCC']:.4f}, AUC: {m_overall['AUC-ROC']:.4f}")
    print(f"    Volatile — Acc: {m_vol['Accuracy']:.4f}, MCC: {m_vol['MCC']:.4f}, AUC: {m_vol['AUC-ROC']:.4f}")
    print(f"    Calm     — Acc: {m_calm['Accuracy']:.4f}, MCC: {m_calm['MCC']:.4f}, AUC: {m_calm['AUC-ROC']:.4f}")
    print(f"    Predictions: {len(y_true_all):,}, Up: {y_pred_all.mean():.1%} [{elapsed:.0f}s]")
    
    all_results.append({
        'model': 'XGBoost (per-stock)', 'feature_set': fs_name,
        **{f'overall_{k}': v for k, v in m_overall.items()},
        **{f'volatile_{k}': v for k, v in m_vol.items()},
        **{f'calm_{k}': v for k, v in m_calm.items()},
        'n_predictions': len(y_true_all), 'time_seconds': elapsed,
    })

print("\nXGBoost complete.")



XGBOOST — WALK-FORWARD (PER-STOCK)

  A_sentiment_only: ['sentiment']
    Done 12/48 months [183s]
    Done 24/48 months [423s]
    Done 36/48 months [713s]
    Done 48/48 months [1037s]
    Overall  — Acc: 0.4998, MCC: -0.0010, AUC: 0.4992
    Volatile — Acc: 0.5002, MCC: -0.0003, AUC: 0.5002
    Calm     — Acc: 0.4995, MCC: -0.0014, AUC: 0.4985
    Predictions: 407,812, Up: 51.1% [1037s]

  B_volatility_only: ['volatility_20d']
    Done 12/48 months [187s]
    Done 24/48 months [422s]
    Done 36/48 months [700s]
    Done 48/48 months [1022s]
    Overall  — Acc: 0.5001, MCC: 0.0006, AUC: 0.4999
    Volatile — Acc: 0.4992, MCC: -0.0018, AUC: 0.4983
    Calm     — Acc: 0.5010, MCC: 0.0023, AUC: 0.5009
    Predictions: 407,812, Up: 49.4% [1022s]

  C_sent_plus_vol: ['sentiment', 'volatility_20d']
    Done 12/48 months [188s]
    Done 24/48 months [423s]
    Done 36/48 months [704s]
    Done 48/48 months [1031s]
    Overall  — Acc: 0.4996, MCC: -0.0010, AUC: 0.4998
    Volatile — Acc: 0.

In [ ]:
# ============================================================
# CELL 10: LSTM Walk-Forward (all 4 feature sets, pooled)
# Estimated time: ~4-8 hours (run overnight)
# ============================================================

print("=" * 70)
print("LSTM — WALK-FORWARD (POOLED, MONTHLY)")
print("=" * 70)

for fs_name, fs_cols in FEATURE_SETS.items():
    print(f"\n  {fs_name}: {fs_cols}")
    
    # Build sequences for this feature set
    print("    Building sequences...")
    X_all, y_all, dates_all, vol_all = create_sequences(model_df, fs_cols, LSTM_BEST['sequence_length'])
    dates_all_pd = pd.to_datetime(dates_all)
    dates_periods = dates_all_pd.to_period('M')
    print(f"    Sequences: {X_all.shape}")
    
    y_true_all, y_pred_all, y_prob_all, y_vol_all_out = [], [], [], []
    t0 = time.time()
    
    for mi, month in enumerate(all_months):
        train_mask = dates_periods < month
        test_mask = dates_periods == month
        
        X_train, y_train = X_all[train_mask], y_all[train_mask]
        X_test, y_test = X_all[test_mask], y_all[test_mask]
        volatile_test = vol_all[test_mask]
        
        if len(X_test) == 0 or len(X_train) == 0:
            continue
        
        # Standardise
        scaler = StandardScaler()
        n_tr, sl, nf = X_train.shape
        X_train_s = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(n_tr, sl, nf)
        X_test_s = scaler.transform(X_test.reshape(-1, nf)).reshape(len(X_test), sl, nf)
        
        # Class weighting
        n_pos = max(y_train.sum(), 1)
        n_neg = max(len(y_train) - n_pos, 1)
        pos_weight = torch.FloatTensor([n_neg / n_pos]).to(device)
        
        # Build and train
        model = LSTMModel(
            input_size=len(fs_cols),
            hidden_size=LSTM_BEST['hidden_size'],
            num_layers=1
        ).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        X_tr_t = torch.FloatTensor(X_train_s).to(device)
        y_tr_t = torch.FloatTensor(y_train).to(device)
        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=1024, shuffle=True)
        
        model.train()
        for epoch in range(LSTM_BEST['epochs']):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                output = model(batch_X).squeeze()
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
        
        # Predict
        model.eval()
        X_test_t = torch.FloatTensor(X_test_s).to(device)
        with torch.no_grad():
            logits = model(X_test_t).squeeze().cpu().numpy()
            probs = 1 / (1 + np.exp(-logits))
        
        preds = (probs > 0.5).astype(int)
        
        y_true_all.extend(y_test)
        y_pred_all.extend(preds)
        y_prob_all.extend(probs)
        y_vol_all_out.extend(volatile_test)
        
        if (mi + 1) % 6 == 0:
            elapsed = time.time() - t0
            pred_up = np.array(y_pred_all).mean()
            print(f"    Done {mi+1}/{len(all_months)} months, up: {pred_up:.1%} [{elapsed:.0f}s]")
    
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    y_vol_all_out = np.array(y_vol_all_out)
    elapsed = time.time() - t0
    
    m_overall = evaluate(y_true_all, y_pred_all, y_prob_all)
    m_vol = evaluate(y_true_all[y_vol_all_out==1], y_pred_all[y_vol_all_out==1], y_prob_all[y_vol_all_out==1])
    m_calm = evaluate(y_true_all[y_vol_all_out==0], y_pred_all[y_vol_all_out==0], y_prob_all[y_vol_all_out==0])
    
    print(f"\n    Overall  — Acc: {m_overall['Accuracy']:.4f}, MCC: {m_overall['MCC']:.4f}, AUC: {m_overall['AUC-ROC']:.4f}")
    print(f"    Volatile — Acc: {m_vol['Accuracy']:.4f}, MCC: {m_vol['MCC']:.4f}, AUC: {m_vol['AUC-ROC']:.4f}")
    print(f"    Calm     — Acc: {m_calm['Accuracy']:.4f}, MCC: {m_calm['MCC']:.4f}, AUC: {m_calm['AUC-ROC']:.4f}")
    print(f"    Predictions: {len(y_true_all):,}, Up: {y_pred_all.mean():.1%} [{elapsed:.0f}s]")
    
    all_results.append({
        'model': 'LSTM (pooled)', 'feature_set': fs_name,
        **{f'overall_{k}': v for k, v in m_overall.items()},
        **{f'volatile_{k}': v for k, v in m_vol.items()},
        **{f'calm_{k}': v for k, v in m_calm.items()},
        'n_predictions': len(y_true_all), 'pred_up_pct': y_pred_all.mean(),
        'time_seconds': elapsed,
    })

print("\nLSTM complete.")

LSTM — WALK-FORWARD (POOLED, MONTHLY)

  A_sentiment_only: ['sentiment']
    Building sequences...
    Sequences: (563213, 30, 1)


In [14]:
import os
import pandas as pd
import time

OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Checking recovered variables...\n")

if "all_results" in globals():
    results_df = pd.DataFrame(all_results)

    save_path = os.path.join(OUTPUT_DIR, "model_rework_recovered_results.csv")
    results_df.to_csv(save_path, index=False)

    print("✅ all_results found")
    print("Rows:", len(results_df))
    print("Saved to:", save_path)
    display(results_df)
else:
    print("❌ all_results not found in memory")

print("\nRecent files in output folder:")
files = sorted(
    [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)],
    key=os.path.getmtime,
    reverse=True
)

for f in files[:20]:
    print(
        time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(f))),
        os.path.basename(f)
    )

Checking recovered variables...

✅ all_results found
Rows: 12
Saved to: ..\..\Dissertation_Results\Prediction\model_rework_recovered_results.csv


,model,feature_set,overall_Accuracy,overall_F1,overall_MCC,overall_AUC-ROC,overall_Pred_Up_Pct,volatile_Accuracy,volatile_F1,volatile_MCC,volatile_AUC-ROC,volatile_Pred_Up_Pct,calm_Accuracy,calm_F1,calm_MCC,calm_AUC-ROC,calm_Pred_Up_Pct,n_predictions,time_seconds,pred_up_pct
0,Logistic Regression,A_sentiment_only,0.499262,0.478719,0.001556,0.500509,0.446474,0.497295,0.497827,-0.003695,0.498493,0.479780,0.500899,0.461533,0.004510,0.501865,0.418736,413753,5.299460,NaN
1,Logistic Regression,B_volatility_only,0.490056,0.436129,-0.014038,0.487288,0.390245,0.493295,0.494894,-0.011886,0.487337,0.481886,0.487357,0.376407,-0.020705,0.486869,0.313924,413753,4.518255,NaN
2,Logistic Regression,C_sent_plus_vol,0.497166,0.490809,-0.004173,0.496766,0.473396,0.495396,0.504691,-0.009001,0.495895,0.497487,0.498640,0.478558,-0.001203,0.497532,0.453333,413753,5.169080,NaN
3,Logistic Regression,D_with_interaction,0.496895,0.492566,-0.004937,0.496193,0.477350,0.495083,0.504272,-0.009610,0.495162,0.497258,0.498405,0.482318,-0.001916,0.497085,0.460770,413753,6.754681,NaN
4,XGBoost (per-stock),A_sentiment_only,0.499823,0.512322,-0.001003,0.499232,0.511424,0.500194,0.514613,-0.000330,0.500223,0.508435,0.499514,0.510398,-0.001435,0.498497,0.513916,407812,1036.684789,NaN
5,XGBoost (per-stock),B_volatility_only,0.500147,0.504251,0.000631,0.499857,0.494073,0.499164,0.510271,-0.001793,0.498293,0.501408,0.500967,0.499100,0.002335,0.500894,0.487959,407812,1021.866314,NaN
6,XGBoost (per-stock),C_sent_plus_vol,0.499613,0.508885,-0.001041,0.499834,0.504676,0.498668,0.514887,-0.003704,0.498667,0.512162,0.500400,0.503749,0.000853,0.500617,0.498435,407812,1031.553032,NaN
7,XGBoost (per-stock),D_with_interaction,0.499659,0.509001,-0.000956,0.499449,0.504821,0.499089,0.514777,-0.002767,0.498313,0.511062,0.500135,0.504068,0.000283,0.500237,0.499618,407812,1054.608094,NaN
8,LSTM (pooled),A_sentiment_only,0.504248,0.555388,0.002873,0.500268,0.600945,0.504175,0.536174,0.004318,0.501796,0.547765,0.504309,0.570204,0.004074,0.500055,0.645191,412196,23381.031702,0.600945
9,LSTM (pooled),B_volatility_only,0.493207,0.508674,-0.014580,0.490966,0.517404,0.497695,0.527708,-0.008240,0.493656,0.542321,0.489473,0.491912,-0.020949,0.486927,0.496673,412196,23470.844305,0.517404



Recent files in output folder:
2026-05-09 12:06:48 model_rework_recovered_results.csv
2026-05-09 09:07:52 prediction_results_final_3x4.csv
2026-05-08 05:54:19 lstm_grid_search_results.csv
2026-05-08 05:34:18 xgb_grid_search_results.csv


In [15]:
# ============================================================
# CELL 11: Final Results Table
# ============================================================

print("=" * 90)
print("FINAL RESULTS: 3 MODELS × 4 FEATURE SETS")
print("=" * 90)

results_df = pd.DataFrame(all_results)

# --- Overall ---
print("\n--- OVERALL ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['overall_Accuracy']:>8.4f} {row['overall_F1']:>8.4f} {row['overall_MCC']:>8.4f} {row['overall_AUC-ROC']:>8.4f}")

# --- Volatile ---
print("\n--- VOLATILE MARKETS ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['volatile_Accuracy']:>8.4f} {row['volatile_F1']:>8.4f} {row['volatile_MCC']:>8.4f} {row['volatile_AUC-ROC']:>8.4f}")

# --- Calm ---
print("\n--- CALM MARKETS ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['calm_Accuracy']:>8.4f} {row['calm_F1']:>8.4f} {row['calm_MCC']:>8.4f} {row['calm_AUC-ROC']:>8.4f}")

# Baseline
test_target = model_df[model_df['date'] >= '2022-01-01']['target']
print(f"\nBaseline (always predict majority class): {test_target.mean():.4f}")

FINAL RESULTS: 3 MODELS × 4 FEATURE SETS

--- OVERALL ---
Model                     Features                       Acc       F1      MCC      AUC
-------------------------------------------------------------------------------------
Logistic Regression       A_sentiment_only            0.4993   0.4787   0.0016   0.5005
Logistic Regression       B_volatility_only           0.4901   0.4361  -0.0140   0.4873
Logistic Regression       C_sent_plus_vol             0.4972   0.4908  -0.0042   0.4968
Logistic Regression       D_with_interaction          0.4969   0.4926  -0.0049   0.4962
XGBoost (per-stock)       A_sentiment_only            0.4998   0.5123  -0.0010   0.4992
XGBoost (per-stock)       B_volatility_only           0.5001   0.5043   0.0006   0.4999
XGBoost (per-stock)       C_sent_plus_vol             0.4996   0.5089  -0.0010   0.4998
XGBoost (per-stock)       D_with_interaction          0.4997   0.5090  -0.0010   0.4994
LSTM (pooled)             A_sentiment_only            0.5042   0

In [17]:
# ============================================================
# CELL 12: Dissertation-Ready Pivot Tables
# ============================================================

print("=" * 90)
print("DISSERTATION TABLE FORMAT")
print("=" * 90)

# Pivot: rows = model, columns = feature set, values = metric
for metric in ['Accuracy', 'MCC', 'AUC-ROC', 'F1']:
    print(f"\n--- {metric} (Overall) ---")
    pivot = results_df.pivot(
        index='model', columns='feature_set', values=f'overall_{metric}'
    )
    pivot = pivot[['A_sentiment_only', 'B_volatility_only', 'C_sent_plus_vol', 'D_with_interaction']]
    pivot.columns = ['Sent', 'Vol', 'Sent+Vol', 'All+Interaction']
    print(pivot.round(4).to_string())

# Volatile vs Calm comparison for feature set D only
print(f"\n--- Feature Set D: Volatile vs Calm ---")
fs_d = results_df[results_df['feature_set'] == 'D_with_interaction']
print(f"{'Model':<25} {'Overall Acc':>12} {'Vol Acc':>10} {'Calm Acc':>10} {'Overall MCC':>12} {'Vol MCC':>10} {'Calm MCC':>10}")
print("-" * 95)
for _, row in fs_d.iterrows():
    print(f"{row['model']:<25} {row['overall_Accuracy']:>12.4f} {row['volatile_Accuracy']:>10.4f} {row['calm_Accuracy']:>10.4f} {row['overall_MCC']:>12.4f} {row['volatile_MCC']:>10.4f} {row['calm_MCC']:>10.4f}")

# Save everything
results_df.to_csv(os.path.join(OUTPUT_DIR, "prediction_results_final_3x4.csv"), index=False)
print(f"\nSaved: prediction_results_final_3x4.csv")

print("\n" + "=" * 90)
print("PIPELINE COMPLETE")
print("=" * 90)
print("""
WHAT TO DO NEXT:
1. Check that all 12 rows are present in results
2. Verify all accuracies are ~50% (confirms prediction failure is universal)
3. Check LSTM pred_up_pct — if any feature set is outside 30-70%, note it
4. Paste outputs back to me for methodology/results writing
""")

DISSERTATION TABLE FORMAT

--- Accuracy (Overall) ---
                       Sent     Vol  Sent+Vol  All+Interaction
model                                                         
LSTM (pooled)        0.5042  0.4932    0.5030           0.4980
Logistic Regression  0.4993  0.4901    0.4972           0.4969
XGBoost (per-stock)  0.4998  0.5001    0.4996           0.4997

--- MCC (Overall) ---
                       Sent     Vol  Sent+Vol  All+Interaction
model                                                         
LSTM (pooled)        0.0029 -0.0146   -0.0008          -0.0094
Logistic Regression  0.0016 -0.0140   -0.0042          -0.0049
XGBoost (per-stock) -0.0010  0.0006   -0.0010          -0.0010

--- AUC-ROC (Overall) ---
                       Sent     Vol  Sent+Vol  All+Interaction
model                                                         
LSTM (pooled)        0.5003  0.4910    0.4982           0.4932
Logistic Regression  0.5005  0.4873    0.4968           0.4962
XGBoost (per-s

In [18]:
# ============================================================
# CELL 12: Complete Results Table
# ============================================================

print("=" * 120)
print("COMPLETE RESULTS: ALL CONFIGURATIONS × ALL METRICS")
print("=" * 120)

# Overall
print("\n--- OVERALL ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['overall_Accuracy']:>8.4f} {row['overall_F1']:>8.4f} {row['overall_MCC']:>8.4f} {row['overall_AUC-ROC']:>8.4f}")

# Volatile
print("\n--- VOLATILE MARKETS ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['volatile_Accuracy']:>8.4f} {row['volatile_F1']:>8.4f} {row['volatile_MCC']:>8.4f} {row['volatile_AUC-ROC']:>8.4f}")

# Calm
print("\n--- CALM MARKETS ---")
print(f"{'Model':<25} {'Features':<25} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 85)
for _, row in results_df.iterrows():
    print(f"{row['model']:<25} {row['feature_set']:<25} {row['calm_Accuracy']:>8.4f} {row['calm_F1']:>8.4f} {row['calm_MCC']:>8.4f} {row['calm_AUC-ROC']:>8.4f}")

# Baseline
test_target = model_df[model_df['date'] >= '2022-01-01']['target']
print(f"\nBaseline (always predict majority class): {test_target.mean():.4f}")

# Save
results_df.to_csv(os.path.join(OUTPUT_DIR, "prediction_results_final_3x4.csv"), index=False)
print(f"\nSaved: prediction_results_final_3x4.csv")

COMPLETE RESULTS: ALL CONFIGURATIONS × ALL METRICS

--- OVERALL ---
Model                     Features                       Acc       F1      MCC      AUC
-------------------------------------------------------------------------------------
Logistic Regression       A_sentiment_only            0.4993   0.4787   0.0016   0.5005
Logistic Regression       B_volatility_only           0.4901   0.4361  -0.0140   0.4873
Logistic Regression       C_sent_plus_vol             0.4972   0.4908  -0.0042   0.4968
Logistic Regression       D_with_interaction          0.4969   0.4926  -0.0049   0.4962
XGBoost (per-stock)       A_sentiment_only            0.4998   0.5123  -0.0010   0.4992
XGBoost (per-stock)       B_volatility_only           0.5001   0.5043   0.0006   0.4999
XGBoost (per-stock)       C_sent_plus_vol             0.4996   0.5089  -0.0010   0.4998
XGBoost (per-stock)       D_with_interaction          0.4997   0.5090  -0.0010   0.4994
LSTM (pooled)             A_sentiment_only            

In [19]:
# VERIFICATION CELL — run this before closing kernel

import pandas as pd
import os

OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"

# Check files exist
files_needed = [
    "prediction_results_final_3x4.csv",
    "xgb_grid_search_results.csv",
    "lstm_grid_search_results.csv",
]

print("=== FILE CHECK ===")
for f in files_needed:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {f} — {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"  ✗ MISSING: {f}")

# Check main results completeness
print("\n=== RESULTS COMPLETENESS ===")
results = pd.read_csv(os.path.join(OUTPUT_DIR, "prediction_results_final_3x4.csv"))

models = results['model'].unique()
feature_sets = results['feature_set'].unique()
print(f"Models: {len(models)} — {list(models)}")
print(f"Feature sets: {len(feature_sets)} — {list(feature_sets)}")
print(f"Total configurations: {len(results)} (expected: 12)")

if len(results) != 12:
    print("  ✗ WARNING: Expected 12 rows")
else:
    print("  ✓ All 12 configurations present")

# Check all metrics exist
expected_metrics = ['overall_Accuracy', 'overall_F1', 'overall_MCC', 'overall_AUC-ROC',
                    'volatile_Accuracy', 'volatile_F1', 'volatile_MCC', 'volatile_AUC-ROC',
                    'calm_Accuracy', 'calm_F1', 'calm_MCC', 'calm_AUC-ROC']
missing = [m for m in expected_metrics if m not in results.columns]
if missing:
    print(f"  ✗ MISSING METRICS: {missing}")
else:
    print(f"  ✓ All {len(expected_metrics)} metrics present")

# Check tuning files
print("\n=== TUNING FILES ===")
xgb_grid = pd.read_csv(os.path.join(OUTPUT_DIR, "xgb_grid_search_results.csv"))
print(f"XGBoost grid: {len(xgb_grid)} combinations")
best_xgb = xgb_grid.sort_values('mcc', ascending=False).iloc[0]
print(f"  Best: n_est={int(best_xgb['n_estimators'])}, depth={int(best_xgb['max_depth'])}, lr={best_xgb['learning_rate']}, MCC={best_xgb['mcc']:.4f}")

lstm_grid = pd.read_csv(os.path.join(OUTPUT_DIR, "lstm_grid_search_results.csv"))
balanced = lstm_grid[(lstm_grid['pred_up_pct'] >= 0.40) & (lstm_grid['pred_up_pct'] <= 0.60)]
print(f"LSTM grid: {len(lstm_grid)} combinations ({len(balanced)} balanced)")
if len(balanced) > 0:
    best_lstm = balanced.sort_values('mcc', ascending=False).iloc[0]
    print(f"  Best: hidden={int(best_lstm['hidden_size'])}, epochs={int(best_lstm['epochs'])}, seq={int(best_lstm['sequence_length'])}, MCC={best_lstm['mcc']:.4f}")
else:
    print("  WARNING: No balanced LSTM configurations")

print("\n=== VERDICT ===")
print("If all checks show ✓, safe to close kernel.")

=== FILE CHECK ===
  ✓ prediction_results_final_3x4.csv — 12 rows, 20 columns
  ✓ xgb_grid_search_results.csv — 27 rows, 7 columns
  ✓ lstm_grid_search_results.csv — 27 rows, 7 columns

=== RESULTS COMPLETENESS ===
Models: 3 — ['Logistic Regression', 'XGBoost (per-stock)', 'LSTM (pooled)']
Feature sets: 4 — ['A_sentiment_only', 'B_volatility_only', 'C_sent_plus_vol', 'D_with_interaction']
Total configurations: 12 (expected: 12)
  ✓ All 12 configurations present
  ✓ All 12 metrics present

=== TUNING FILES ===
XGBoost grid: 27 combinations
  Best: n_est=50, depth=5, lr=0.05, MCC=0.0062
LSTM grid: 27 combinations (8 balanced)
  Best: hidden=128, epochs=15, seq=30, MCC=-0.0035

=== VERDICT ===
If all checks show ✓, safe to close kernel.


In [20]:
# VOLATILITY DECILE ANALYSIS — Quick check, LR only (fastest)
# Run this as a standalone cell after loading data

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
import os

DATA_PATH = r"..\..\Data\Main\modelling_dataset.csv"
OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
model_df = df.dropna(subset=['volatility_20d', 'next_day_return']).copy()
model_df = model_df.sort_values(['ticker', 'date']).reset_index(drop=True)
model_df['year_month'] = model_df['date'].dt.to_period('M')

features = ['sentiment', 'volatility_20d', 'sent_x_vol']
all_months = sorted(model_df[model_df['date'] >= '2022-01-01']['year_month'].unique())

# Compute volatility deciles on FULL dataset
model_df['vol_decile'] = pd.qcut(model_df['volatility_20d'], 10, labels=['D1','D2','D3','D4','D5','D6','D7','D8','D9','D10'])

print("Volatility decile cutoffs:")
for d in ['D1','D2','D3','D4','D5','D6','D7','D8','D9','D10']:
    subset = model_df[model_df['vol_decile'] == d]
    print(f"  {d}: vol {subset['volatility_20d'].min():.4f} – {subset['volatility_20d'].max():.4f} ({len(subset):,} rows)")

# Walk-forward LR, store predictions with decile labels
y_true_all, y_pred_all, y_prob_all, y_decile_all = [], [], [], []

print("\nRunning LR walk-forward...")
for mi, month in enumerate(all_months):
    train = model_df[model_df['year_month'] < month]
    test = model_df[model_df['year_month'] == month]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features])
    X_test = scaler.transform(test[features])
    
    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr.fit(X_train, train['target'].values)
    
    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]
    
    y_true_all.extend(test['target'].values)
    y_pred_all.extend(y_pred)
    y_prob_all.extend(y_prob)
    y_decile_all.extend(test['vol_decile'].values)

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_prob_all = np.array(y_prob_all)
y_decile_all = np.array(y_decile_all)

print(f"Total predictions: {len(y_true_all):,}")

# Compute metrics per decile
print(f"\n{'Decile':<8} {'N':>8} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8} {'Up%':>8}")
print("-" * 58)

decile_results = []
for d in ['D1','D2','D3','D4','D5','D6','D7','D8','D9','D10']:
    mask = y_decile_all == d
    yt = y_true_all[mask]
    yp = y_pred_all[mask]
    ypr = y_prob_all[mask]
    
    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, zero_division=0)
    mcc = matthews_corrcoef(yt, yp)
    try:
        auc = roc_auc_score(yt, ypr)
    except:
        auc = np.nan
    up_pct = yt.mean()
    
    print(f"{d:<8} {mask.sum():>8,} {acc:>8.4f} {f1:>8.4f} {mcc:>8.4f} {auc:>8.4f} {up_pct:>7.1%}")
    
    decile_results.append({
        'decile': d, 'n': mask.sum(), 'accuracy': acc, 'f1': f1, 
        'mcc': mcc, 'auc_roc': auc, 'actual_up_pct': up_pct
    })

decile_df = pd.DataFrame(decile_results)
decile_df.to_csv(os.path.join(OUTPUT_DIR, "volatility_decile_results.csv"), index=False)
print(f"\nSaved: volatility_decile_results.csv")

Volatility decile cutoffs:
  D1: vol 0.0013 – 0.0107 (57,891 rows)
  D2: vol 0.0107 – 0.0127 (57,890 rows)
  D3: vol 0.0127 – 0.0145 (57,890 rows)
  D4: vol 0.0145 – 0.0163 (57,890 rows)
  D5: vol 0.0163 – 0.0183 (57,891 rows)
  D6: vol 0.0183 – 0.0205 (57,890 rows)
  D7: vol 0.0205 – 0.0233 (57,890 rows)
  D8: vol 0.0233 – 0.0274 (57,890 rows)
  D9: vol 0.0274 – 0.0349 (57,890 rows)
  D10: vol 0.0349 – 0.2870 (57,891 rows)

Running LR walk-forward...
Total predictions: 413,753

Decile          N      Acc       F1      MCC      AUC      Up%
----------------------------------------------------------
D1         44,320   0.4939   0.4786  -0.0101   0.4929   51.3%
D2         44,918   0.4927   0.4851  -0.0133   0.4902   51.3%
D3         44,984   0.4916   0.4924  -0.0153   0.4909   52.0%
D4         44,075   0.4925   0.4938  -0.0142   0.4918   51.6%
D5         42,866   0.4979   0.5017  -0.0037   0.4983   51.5%
D6         42,511   0.4966   0.5047  -0.0067   0.4989   51.7%
D7         41,931   0.